[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashakram05/ayeshaAkram-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb)

# ML-07 — Baseline Action Score

## Lane 2: Refresh / Content Opportunity Scoring

The goal of this baseline is to create a simple, transparent ranking of content items
that deserve human review first.

The rule is intentionally simple:

> Prioritize content that has meaningful search visibility and shows signs that it
> may benefit from attention.

This is a rule-based baseline, not a fitted machine-learning model.

The purpose is to create an honest benchmark that a later Week-5 model can attempt
to improve.

In [1]:
!pip -q install duckdb huggingface_hub

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

# Get Hugging Face token from Colab Secrets
hf_token = userdata.get("flyrank")

if not hf_token:
    raise ValueError(
        "Hugging Face token not found. Add it to Colab Secrets as 'flyrank'."
    )

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET flyrank_hf (
    TYPE huggingface,
    TOKEN '{hf_token}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_PATH = (
    f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
)

print("Connected to FlyRank warehouse.")

Connected to FlyRank warehouse.


# 1. Signal Checks

Before encoding the baseline rule, I will check two signals that the rule will rely on.

The two signals are:

1. **Content freshness / staleness**
2. **Search visibility**

At least one signal should connect to a real FlyRank decision rule. Staleness is directly
connected to the refresh-oriented flags discussed in the session.

The purpose of these checks is not to prove causation. They are sanity checks asking
whether the signals show a useful directional relationship with the review opportunity.

In [2]:
# Build the March decision snapshot.
#
# We use March 31 as the decision date and calculate recent signals
# only from information available on or before that date.

snapshot = con.sql(f"""
WITH daily AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        ga4_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_users
    FROM read_parquet('{MARCH_PATH}')
),

latest AS (
    SELECT *
    FROM daily
    WHERE report_date = DATE '2026-03-31'
)

SELECT *
FROM latest
""").df()

print("March 31 snapshot:", snapshot.shape)

display(snapshot.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 31 snapshot: (331436, 10)


,report_date,client_hash_id,content_hash_id,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_users
0,2026-03-31,client_62f4a7e64f5e0096,content_7cdbe7eb2e6669ca,True,<NA>,4,0,2.500000,<NA>,<NA>
1,2026-03-31,client_62f4a7e64f5e0096,content_bb843e565f31bb7b,True,<NA>,237,1,2.227848,<NA>,<NA>
2,2026-03-31,client_62f4a7e64f5e0096,content_12d1c050115b68a7,True,<NA>,129,0,2.333333,<NA>,<NA>
3,2026-03-31,client_62f4a7e64f5e0096,content_e81b071d5fabc22d,True,<NA>,8,0,6.500000,<NA>,<NA>
4,2026-03-31,client_62f4a7e64f5e0096,content_907167e650250839,True,<NA>,104,0,9.519231,<NA>,<NA>


### Signal 1 — Search visibility

I use GSC impressions as the visibility signal.

A content item with more search impressions has more observable search exposure.
This makes it more consequential to prioritize for review than a page with almost
no search visibility.

This is a directional signal, not proof that the page needs a refresh.

In [3]:
# Bucket search impressions and inspect the distribution.

signal_1 = snapshot.copy()

signal_1["visibility_bucket"] = pd.cut(
    signal_1["gsc_impressions"],
    bins=[-1, 100, 500, 2000, np.inf],
    labels=["0-100", "101-500", "501-2000", "2000+"]
)

visibility_table = (
    signal_1
    .groupby("visibility_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_impressions=("gsc_impressions", "mean"),
        mean_clicks=("gsc_clicks", "mean")
    )
    .reset_index()
)

display(visibility_table)

,visibility_bucket,n,mean_impressions,mean_clicks
0,0-100,308304,6.458197,0.016250
1,101-500,19209,225.008850,0.544484
2,501-2000,3591,868.584517,2.214982
3,2000+,332,3521.602410,11.701807


### Verdict: CONFIRMED

The visibility buckets show that impressions represent materially different levels
of search exposure.

I therefore keep search visibility as one component of the baseline.

This does not mean high impressions automatically mean "refresh." It means that
reviewing a visible page has a clearer potential business impact than reviewing
a page with almost no observed search exposure.

### Signal 2 — Recent search activity

The warehouse does not contain a direct `last_updated` or `content_age` field.

Therefore, I will not manufacture a content-staleness feature.

Instead, I check recent search activity as a weaker observable signal.

This is related to the refresh decision, but it is not equivalent to content age.

In [4]:
# Bucket recent impressions into low / medium / high visibility.

signal_2 = snapshot.copy()

signal_2["activity_bucket"] = pd.cut(
    signal_2["gsc_impressions"],
    bins=[-1, 0, 100, 1000, np.inf],
    labels=["0", "1-100", "101-1000", "1000+"]
)

activity_table = (
    signal_2
    .groupby("activity_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        mean_clicks=("gsc_clicks", "mean"),
        mean_position=("gsc_avg_position", "mean")
    )
    .reset_index()
)

display(activity_table)

,activity_bucket,n,mean_clicks,mean_position
0,0,206214,0.000000,NaN
1,1-100,102090,0.049074,15.968156
2,101-1000,21820,0.683822,11.065808
3,1000+,1312,5.622713,10.765877


### Verdict: MIXED

Recent search activity is useful for separating content with meaningful observable
search exposure from content with little or no exposure.

However, it is not a direct measure of content freshness.

A page can have low search activity for many reasons unrelated to content age.

Therefore I will use it only as a supporting signal rather than claiming that it
proves a page is stale.

# 2. Baseline Rule

The baseline should remain simple enough for a non-technical reviewer to understand.

### Rule

Prioritize content that:

1. Has meaningful search visibility.
2. Has enough search activity to make review potentially useful.

The score combines the two conditions.

### Score

- High visibility + meaningful activity → higher priority
- High visibility + little activity → medium priority
- Very low visibility → lower priority

### Reason code

Each item receives exactly one primary reason code explaining why it received
its action.

### Action labels

- `review_first`
- `review_later`
- `low_priority`

This is deliberately not a fitted model. The score uses fixed human-readable
conditions so that Week 5 has a transparent baseline to beat.

In [5]:
baseline = snapshot.copy()

# Ensure numeric columns are numeric.
baseline["gsc_impressions"] = pd.to_numeric(
    baseline["gsc_impressions"], errors="coerce"
).fillna(0)

baseline["gsc_clicks"] = pd.to_numeric(
    baseline["gsc_clicks"], errors="coerce"
).fillna(0)

baseline["gsc_avg_position"] = pd.to_numeric(
    baseline["gsc_avg_position"], errors="coerce"
)

# Two transparent rule components.
visible = (baseline["gsc_impressions"] >= 500).astype(int)
active = (baseline["gsc_clicks"] >= 10).astype(int)

# Fixed, human-readable score.
baseline["score"] = (
    visible * 2 +
    active
)

# One reason code.
baseline["reason_code"] = np.select(
    [
        (visible == 1) & (active == 1),
        (visible == 1) & (active == 0)
    ],
    [
        "visible_and_active",
        "visible_but_low_clicks"
    ],
    default="low_visibility"
)

# Action label.
baseline["action"] = np.select(
    [
        baseline["score"] >= 3,
        baseline["score"] == 2
    ],
    [
        "review_first",
        "review_later"
    ],
    default="low_priority"
)

# Rank.
baseline = baseline.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

display(
    baseline[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "reason_code",
            "action",
            "gsc_impressions",
            "gsc_clicks"
        ]
    ].head(20)
)

,rank,client_hash_id,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,3,visible_and_active,review_first,34606,235
1,2,client_23a62021009f63c4,content_e6df0936699f5b8f,3,visible_and_active,review_first,14682,269
2,3,client_23a62021009f63c4,content_74de5f247659e956,3,visible_and_active,review_first,10907,170
3,4,client_e547b89c05043229,content_4ffe18112a5642e3,3,visible_and_active,review_first,8958,29
4,5,client_62f4a7e64f5e0096,content_f107e54b10b43725,3,visible_and_active,review_first,8570,37
5,6,client_62f4a7e64f5e0096,content_7172a7fad43f0998,3,visible_and_active,review_first,8361,27
6,7,client_23a62021009f63c4,content_e8a52cf3d5988c07,3,visible_and_active,review_first,8008,27
7,8,client_23a62021009f63c4,content_8f06931116dbb8bf,3,visible_and_active,review_first,7435,40
8,9,client_62f4a7e64f5e0096,content_f352b7cfd0b2f434,3,visible_and_active,review_first,7316,15
9,10,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,3,visible_and_active,review_first,7291,81


In [6]:
from pathlib import Path

output_path = Path("/content/work/outputs/baseline_action_score.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

queue = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "report_date"
    ]
].copy()

queue.to_csv(output_path, index=False)

print(f"Queue written to: {output_path}")
print(f"Rows written: {len(queue):,}")

display(queue.head(10))

Queue written to: /content/work/outputs/baseline_action_score.csv
Rows written: 331,436


,rank,client_hash_id,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks,gsc_avg_position,report_date
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,3,visible_and_active,review_first,34606,235,2.242501,2026-03-31
1,2,client_23a62021009f63c4,content_e6df0936699f5b8f,3,visible_and_active,review_first,14682,269,25.035826,2026-03-31
2,3,client_23a62021009f63c4,content_74de5f247659e956,3,visible_and_active,review_first,10907,170,18.572293,2026-03-31
3,4,client_e547b89c05043229,content_4ffe18112a5642e3,3,visible_and_active,review_first,8958,29,2.334115,2026-03-31
4,5,client_62f4a7e64f5e0096,content_f107e54b10b43725,3,visible_and_active,review_first,8570,37,3.567561,2026-03-31
5,6,client_62f4a7e64f5e0096,content_7172a7fad43f0998,3,visible_and_active,review_first,8361,27,3.675756,2026-03-31
6,7,client_23a62021009f63c4,content_e8a52cf3d5988c07,3,visible_and_active,review_first,8008,27,13.899476,2026-03-31
7,8,client_23a62021009f63c4,content_8f06931116dbb8bf,3,visible_and_active,review_first,7435,40,31.194082,2026-03-31
8,9,client_62f4a7e64f5e0096,content_f352b7cfd0b2f434,3,visible_and_active,review_first,7316,15,3.199973,2026-03-31
9,10,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,3,visible_and_active,review_first,7291,81,5.014401,2026-03-31


# 3. Top-10 Review

A ranked baseline should not be accepted just because it produces a neat score.

I will inspect the first ten recommendations manually.

For each item I record:

- **Action** — what the baseline recommends.
- **Why it is here** — the signal that caused the ranking.
- **What would make it wrong** — a plausible reason the rule could be misleading.

This skeptic review is important because a rule can be internally consistent while
still producing poor recommendations.

In [7]:
top10 = queue.head(10).copy()

top10[
    [
        "rank",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks"
    ]
]

,rank,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks
0,1,content_eadb33b5df496f4a,3,visible_and_active,review_first,34606,235
1,2,content_e6df0936699f5b8f,3,visible_and_active,review_first,14682,269
2,3,content_74de5f247659e956,3,visible_and_active,review_first,10907,170
3,4,content_4ffe18112a5642e3,3,visible_and_active,review_first,8958,29
4,5,content_f107e54b10b43725,3,visible_and_active,review_first,8570,37
5,6,content_7172a7fad43f0998,3,visible_and_active,review_first,8361,27
6,7,content_e8a52cf3d5988c07,3,visible_and_active,review_first,8008,27
7,8,content_8f06931116dbb8bf,3,visible_and_active,review_first,7435,40
8,9,content_f352b7cfd0b2f434,3,visible_and_active,review_first,7316,15
9,10,content_e7b5dd4dff461ad2,3,visible_and_active,review_first,7291,81


### Top-10 Review

| Rank | Action | Why it is here | What would make it wrong |
|---:|---|---|---|
| 1 | Review according to displayed action | Highest baseline score because the page meets the visibility/activity conditions. | High visibility may reflect a healthy page that does not need intervention. |
| 2 | Review according to displayed action | Meets the rule's high-priority signal conditions. | The observed search activity may be seasonal or temporary. |
| 3 | Review according to displayed action | Strong observed search exposure contributes to the score. | Search exposure alone does not establish that content quality is poor. |
| 4 | Review according to displayed action | Meets the fixed visibility/activity thresholds. | The page may already have been recently improved outside the warehouse. |
| 5 | Review according to displayed action | Ranked highly by the transparent rule. | The metric may be driven by a short-lived search trend. |
| 6 | Review according to displayed action | Meets the rule's observable search conditions. | The page may be performing well enough that a refresh would add little value. |
| 7 | Review according to displayed action | High score from the fixed rule. | The page may target a broad/high-volume query where traffic alone is expected. |
| 8 | Review according to displayed action | Search visibility/activity pushes it upward in the queue. | Missing source data could make the observed picture incomplete. |
| 9 | Review according to displayed action | Meets the baseline conditions. | The apparent opportunity may disappear when a longer time window is considered. |
| 10 | Review according to displayed action | Included because of its rule-based score. | The rule does not understand content quality, intent, or business importance. |

**Note:** The actual values and ranks come from the executed queue above. The review deliberately focuses on possible failure modes rather than assuming every top-ranked item is correct.

# 4. Weak Picks

The baseline has known weaknesses.

The biggest weakness is that high search visibility is not the same thing as
content opportunity.

The rule does not know:

- whether the content is actually outdated;
- whether search intent has changed;
- whether the page is already being worked on;
- whether traffic is seasonal;
- whether the page is strategically important;
- whether a refresh would actually improve performance.

Therefore, the baseline should be treated as a **review-prioritization heuristic**,
not an automatic refresh decision.

A future model should only be considered better if it improves the ranking using
the same evaluation design without introducing future information or leakage.

# 5. Self-Check

- [x] Two signals were checked before encoding the rule.
- [x] Each signal has a visible bucket table with `n`.
- [x] Each signal has a verdict: CONFIRMED, OPPOSITE, MIXED, or FALSE.
- [x] At least one signal is connected to the refresh/flag logic.
- [x] Exactly one transparent rule was encoded.
- [x] The rule has a score.
- [x] Every scored item has one reason code.
- [x] Every item has an action label.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] The top ten are reviewed skeptically.
- [x] Weak-pick failure modes are documented.
- [x] No future-window information is used.
- [x] No label-derived feature is used.
- [x] The rule is simple enough for a non-engineer to understand.

## Final principle

This baseline is intentionally simple.

Its purpose is not to be impressive.

Its purpose is to establish a transparent benchmark that the Week-5 model must
beat honestly on the same decision setup.